### General settings

In [1]:
#Change these values please :)
mail = "bianchigianpaolo2@gmail.com"
user_name = "Pastasciutta"

from google.colab import drive
import os
import sys

drive.mount('/content/gdrive/')

# Define the path to the directory containing your package
package_parent_dir = '/content/gdrive/MyDrive/Colab Notebooks'

# Append to sys.path if it is not already present
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

# Verify the path was added
print(sys.path)

#Import client
from millionaire_client import MillionaireClient, AuthenticationError

#Get password
from google.colab import userdata
pwd = userdata.get('poli-millionaire')

#Login
API_URL = "http://131.175.15.22:51111/"
username = user_name
password = pwd
client = MillionaireClient(API_URL)
try:
    user = client.login(username, password)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")

Mounted at /content/gdrive/
['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/gdrive/MyDrive/Colab Notebooks']

Welcome, Pastasciutta! (Role: student)


In [2]:
def play_game(game, comp_id):
  # Play the game
  start_time = time.time()

  # Lists to store metrics for each question in this game
  question_search_doc_times = []
  question_prefilter_times = []
  question_rerank_doc_times = []
  question_reasoning_times = []

  num_questions_answered = 0
  num_correct_answers = 0

  while game.in_progress:
      question = game.current_question
      if not question:
          print("No question available. Game may have ended.")
          break

      print(f"\n--- Level {game.current_level} ---")
      print(f"Q: {question.text}")
      for opt in question.options:
        print(f"{opt.id}: {opt.text}")
      print()

      # Infer answer and get metrics
      response_id, metrics = infer_answer(question, comp_id)
      print(f"Selected answer: {response_id}")

      # Store metrics for this question
      question_search_doc_times.append(metrics["search_doc_time"])
      question_prefilter_times.append(metrics["prefilter_time"])
      question_rerank_doc_times.append(metrics["rerank_doc_time"])
      question_reasoning_times.append(metrics["reasoning_time"])

      time_before_answer = game.time_remaining
      result = game.answer(response_id)
      time_to_answer = time.time() - start_time

      num_questions_answered += 1

      if result.correct:
          print(" CORRECT!")
          num_correct_answers += 1
          if result.game_over:
              print(f"\n CONGRATULATIONS! You completed the game!")
              print(f" Final earnings: ${result.earned_amount:,.2f}")
          else:
              print(f" Earned so far: ${result.earned_amount:,.2f}")
      elif result.timed_out:
        print("TIMED OUT!")
        print(f"\n Game Over!")
        print(f" Final earnings: ${result.earned_amount:,.2f}")
        # Even if timed out, capture metrics up to this point
        break # Exit loop as game is over
      elif not result.correct:
          print(" WRONG ANSWER!")
          print(f"\n Game Over!")
          print(f" Final earnings: ${result.earned_amount:,.2f}")
          break # Exit loop as game is over

  print("\n=== Game Summary ===")
  print(f"Reached Level: {game.current_level}")
  print(f"Total Earnings: ${game.earned_amount:,.2f}")

  avg_response_time = time_to_answer / num_questions_answered if num_questions_answered > 0 else 0
  avg_search_doc_time = sum(question_search_doc_times) / num_questions_answered if num_questions_answered > 0 else 0
  avg_prefilter_time = sum(question_prefilter_times) / num_questions_answered if num_questions_answered > 0 else 0
  avg_rerank_doc_time = sum(question_rerank_doc_times) / num_questions_answered if num_questions_answered > 0 else 0
  avg_reasoning_time = sum(question_reasoning_times) / num_questions_answered if num_questions_answered > 0 else 0

  accuracy_per_run = num_correct_answers / num_questions_answered if num_questions_answered > 0 else 0

  game_metrics = {
      "level_reached": game.current_level,
      "avg_response_time": avg_response_time,
      "avg_search_doc_time": avg_search_doc_time,
      "avg_prefilter_time": avg_prefilter_time,
      "avg_rerank_doc_time": avg_rerank_doc_time,
      "avg_reasoning_time": avg_reasoning_time,
      "accuracy_per_run": accuracy_per_run
  }

  return game_metrics


# RAG model

Model settings


In [3]:
!pip install -U transformers accelerate bitsandbytes
!pip install sentence-transformers
!pip install rank-bm25
from sentence_transformers import SentenceTransformer, util, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from concurrent.futures import ThreadPoolExecutor
from rank_bm25 import BM25Okapi
import torch
import re
import random
import time
import requests
import numpy as np

HEADERS = {
    "User-Agent": (
        "WhoWantsToBeAMillionaire-Bot/1.0 (research project;"
        " bianchigianpaolo2@gmail.com)"
    )
}

#embedder = SentenceTransformer(
#    "all-MiniLM-L6-v2",
#    device="cpu"
#)

reranker = CrossEncoder(
    "BAAI/bge-reranker-base",
    device="cpu"
)


def load_model(model_name):
    global current_model, current_tokenizer, current_name
    if current_name == model_name:
        return

    if current_model is not None:
        del current_model
        torch.cuda.empty_cache()

    current_tokenizer = AutoTokenizer.from_pretrained(model_name)

    current_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quant_config,
        device_map="auto",
        torch_dtype=torch.float16
    )

    current_name = model_name
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
current_model = None
current_tokenizer = None
current_name = None
rag_model = "Qwen/Qwen2.5-14B-Instruct"
math_model = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

In [4]:
if torch.cuda.is_available():
    print("CUDA is available! Using GPU.")
    print(f"Current device: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is not available. Using CPU.")


CUDA is available! Using GPU.
Current device: Tesla T4


## Functions

In [5]:
# ==============================================================================
# PROMPT BUILDERS
# ==============================================================================

#From some previous experiments, actually adding the theme of the questions may result in lower performance (extra tokens confusing the model or biasing the type of query)
def build_query_prompt(question, options):
    system_prompt = {
        "role": "system",
        "content": f"""You are a Wikipedia search assistant. Given a multiple choice question, output EXACTLY three Wikipedia search queries, one per line, nothing else.

Rules:
- Write short keyword phrases, not full sentences.
- Target Wikipedia article titles, named entities, historical events, or scientific concepts.
- Each query must be independently searchable and cover a different angle of the question.
- No filler words: avoid "what is", "explain", "who was", "describe".
- No numbering, no bullets, no extra text.

Example:
QUESTION: What element has atomic number 79?
A) Silver  B) Gold  C) Platinum  D) Copper
OUTPUT:
Gold chemical element
Atomic number periodic table
Noble metals chemistry""",
    }

    user_prompt = {
        "role": "user",
        "content": f"""#QUESTION: {question}

#OPTIONS:
A) {options[0]}
B) {options[1]}
C) {options[2]}
D) {options[3]}""",
    }

    return [system_prompt, user_prompt]


def build_doc_prompt(question, options, documents):
    system_prompt = {
        "role": "system",
        "content": """You are an expert quiz player.

Use the documents as evidence.
Reason silently.

Output exactly one line:

FINAL_ANSWER: X

where X is A, B, C, or D.""",
    }

    formatted_docs = "\n\n".join(
        f"[Document {i+1}]\n{doc}" for i, doc in enumerate(documents)
    )
    print(formatted_docs)

    user_prompt = {
        "role": "user",
        "content": f"""#QUESTION: {question}
#OPTIONS:
A) {options[0]}
B) {options[1]}
C) {options[2]}
D) {options[3]}

#REFERENCE DOCUMENTS:
{formatted_docs}""",
    }

    return [system_prompt, user_prompt]

#Numerous configurations have been tried: both letting the model think and then output or simply output
#By letting the model think, we are not able to enforce the 30s time budget, as we are not sure to receive an answer before that time.
#Moreover, in thinking mode, we cannot enforce the model to make checkpoints (like write CURRENT_BEST) as there are no constraints for the thinking part.
#Also, putting a token constraint would also incur in the same problem.
#This solution simply tells the model to OUTPUT within a given sentence limit (often not met though) while making sure the model "think:False"
def build_math_prompt(question, options):
    system = {
        "role": "system",
        "content": """
You are a mathematics and statistics expert solving timed multiple-choice questions.

Rules:
- Reason carefully but concisely.
- Use short, precise steps.
- Verify definitions and theorem statements exactly.
- Do not guess based on intuition alone.
- Avoid unnecessary explanations.
- Stop once the answer is determined.

Output format:

Key steps:
- ...

FINAL_ANSWER: X

Where X is A, B, C, or D.
"""
    }

    user = {
        "role": "user",
        "content": f"""Question:
{question}

Options:
A) {options[0]}
B) {options[1]}
C) {options[2]}
D) {options[3]}"""
    }

    return [system, user]

# ==============================================================================
# TEXT PROCESSING & CHUNKING HELPERS
# ==============================================================================

#Splits Wikipedia by paragraphs (following \n\n).
#Appends sentences until max_words is reached (stops actually at a dot to avoid cutting off paragraphs).
def chunk_text(text, max_words=250):
    paragraphs = text.split("\n\n")
    chunks = []

    for p in paragraphs:
        p = p.strip()
        if not p:
            continue

        words = p.split()
        if len(words) <= max_words:
            chunks.append(p)
            continue

        # Split paragraph into sentences
        sentences = re.split(r'(?<=[.!?])\s+', p)

        current_chunk = []
        current_word_count = 0

        for sentence in sentences:
            sentence_words = sentence.split()
            sentence_word_count = len(sentence_words)

            if current_word_count + sentence_word_count > max_words and current_chunk:
                # Current chunk is full — flush it
                chunks.append(" ".join(current_chunk))
                current_chunk = sentence_words
                current_word_count = sentence_word_count
            else:
                current_chunk.extend(sentence_words)
                current_word_count += sentence_word_count

        # Don't forget the last chunk
        if current_chunk:
            chunks.append(" ".join(current_chunk))

    return chunks

#Cleans a query
#This was done because it was usual for the model to output strange values, such as chinese characters, words stitched together, long sentences...
#BUG sometimes those still happen 0_0
def clean_query(q):
    q = q.strip().replace('"', "")
    q = q.split("\n")[0]  # first line only
    q = re.sub(r"[^a-zA-Z0-9\s\-']", " ", q) #leave only letter, digit, whitespace, hyphen or apostrophe with a space
    q = re.sub(r"\s+", " ", q).strip()
    return q[:120]


# ==============================================================================
# RERANKING & RETRIEVAL (RAG Core)
# ==============================================================================


def get_top_chunks(question, options, chunks, top_k=5):
    # Embed question and each option separately
    start_time = time.time()
    question_embedding = embedder.encode(question)
    option_embeddings = embedder.encode(options)
    chunk_embeddings = embedder.encode(chunks)

    # Normalize everything
    chunks_norm = chunk_embeddings / np.linalg.norm(
        chunk_embeddings, axis=1, keepdims=True
    )
    question_norm = question_embedding / np.linalg.norm(question_embedding)
    options_norm = option_embeddings / np.linalg.norm(
        option_embeddings, axis=1, keepdims=True
    )

    # Similarity with question
    question_sim = chunks_norm @ question_norm

    # Max similarity across all options (not average — max preserves signal)
    option_sims = chunks_norm @ options_norm.T  # shape: (n_chunks, 4)
    best_option_sim = option_sims.max(axis=1)  # best matching option per chunk

    # Combine: weight question more heavily than options
    combined = 0.6 * question_sim + 0.4 * best_option_sim

    top_indices = np.argsort(combined)[::-1][:top_k]
    chunk_time = time.time() - start_time
    #print(f"Time to chunk documents {chunk_time:.1f}")
    return [chunks[i] for i in top_indices], chunk_time


#Takes the best chunks
#In particular it takes all 4 possible [question,option] pairs and scores all chunks against that pair. Then the best <top_k> chunks are taken
def rerank_chunks_v2(question, options, chunks, top_k=3):
    best_chunks = []

    for option in options:
        query = f"{question} {option}"
        pairs = [[query, chunk] for chunk in chunks]
        scores = reranker.predict(pairs)

        best_idx = np.argmax(scores)
        best_chunks.append((scores[best_idx], chunks[best_idx]))

    best_chunks.sort(key=lambda x: x[0], reverse=True)

    seen = set()
    final = []
    for score, chunk in best_chunks:
        norm = re.sub(r"\s+", " ", chunk.strip().lower())
        if norm not in seen:
            seen.add(norm)
            final.append(chunk)
        if len(final) == top_k:
            break

    return final

def rerank_chunks_v3(question, options, chunks, top_k=3):
    # Single query combining question + all options
    start_time = time.time()
    query = question + " " + " ".join(options)

    pairs = [[query, chunk] for chunk in chunks]
    scores = reranker.predict(pairs)

    # Rank all chunks, take top_k directly
    ranked = sorted(zip(scores, chunks), key=lambda x: x[0], reverse=True)

    seen = set()
    final = []
    for score, chunk in ranked:
        norm = re.sub(r"\s+", " ", chunk.strip().lower())
        if norm not in seen:
            seen.add(norm)
            final.append(chunk)
        if len(final) == top_k:
            break
    rerank_time = time.time() - start_time
    #print(f"Time to rerank documents {rerank_time:.1f}")
    return final, rerank_time

def bm25_prefilter(question, options, chunks, top_k=10):
    start_time = time.time()
    # Build query from question + all options combined
    query = f"{question} {' '.join(options)}"
    tokenized_query = query.lower().split()
    tokenized_chunks = [c.lower().split() for c in chunks]

    bm25 = BM25Okapi(tokenized_chunks)
    scores = bm25.get_scores(tokenized_query)

    ranked = sorted(zip(scores, chunks), key=lambda x: x[0], reverse=True)
    prefilter_time = time.time() - start_time
    #print(f"Time to bm25 prefilter documents {prefilter_time:.3f}")
    return [chunk for _, chunk in ranked[:top_k]], prefilter_time

#Performs a single wikipedia search API call
#It takes a query and returns the <srlimit> best possible titles/pages. It then extracts the full content of those pages.
#So if we have 3 queries generated, it will take 6 total pages.
def wikisearch_single(query):
    search_url = "https://en.wikipedia.org/w/api.php"
    search_params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json",
        "srlimit": 2,
    }
    search_response = requests.get(
        search_url, params=search_params, headers=HEADERS
    )

    # No titles or rate limited, a log could be made here
    if not search_response.text or search_response.status_code != 200:
        return []

    titles = [r["title"] for r in search_response.json()["query"]["search"]]
    print(f"Titles found for query {query}: {titles}")

    pages = []
    # downloading full wikipedia page
    for title in titles:
        extract_params = {
            "action": "query",
            "titles": title,
            "prop": "extracts",
            "explaintext": True,
            "format": "json",
        }
        extract_response = requests.get(
            search_url, params=extract_params, headers=HEADERS
        )

        # No documents or rate limited, a log could be made here
        if not extract_response.text or extract_response.status_code != 200:
            continue

        for page in extract_response.json()["query"]["pages"].values():
            if "extract" in page:
                pages.append((title, page["extract"]))

    return pages

#Performs multiple wikipedia search API calls in parallel
def wikisearch_multi(queries, question, options):

    def search_one(query):
        #query = query.strip().replace('"', "")
        if not query:
            return []
        return wikisearch_single(query)

    start_time = time.time()
    with ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(search_one, queries))

    # Pool all chunks from all queries
    all_pages = [page for query_pages in results for page in query_pages]

    # Now deduplicate by title
    seen_titles = set()
    unique_pages = []
    for title, text in all_pages:
        if title not in seen_titles:
            seen_titles.add(title)
            unique_pages.append((title, text))

    # Now chunk the unique full texts
    # Title prefix gives the reranker article-level context at zero cost
    all_chunks = []
    chunk_doc_time = 0
    for title, text in unique_pages:
        chunks_for_page = chunk_text(text)
        for chunk in chunks_for_page:
            all_chunks.append(f"[{title}] {chunk}")

    if not all_chunks:
        return [], 0, 0, 0

    search_doc_time = time.time()-start_time
    #print(f"Time to search the documents {search_doc_time:.1f}")

    prefiltered_chunks, prefilter_time = bm25_prefilter(question, options, all_chunks, top_k=10)
    final_chunks, rerank_time = rerank_chunks_v3(question, options, prefiltered_chunks, top_k=5)

    # For simplicity, let's assume chunk_text time is negligible or we can capture it more precisely later if needed.
    # For now, we'll return search_doc_time, prefilter_time, and rerank_time
    return final_chunks, search_doc_time, prefilter_time, rerank_time


# ==============================================================================
# MODEL INFERENCE CALLS (LLM Engine)
# ==============================================================================

def call_model(prompt, temp, thinking, max_new_tokens=512):
    text = current_tokenizer.apply_chat_template(
        prompt,
        tokenize=False,
        think=thinking,
        add_generation_prompt=True
    )

    inputs = current_tokenizer(
        text,
        return_tensors="pt"
    ).to("cuda")

    input_length = inputs["input_ids"].shape[1]  # store input length

    outputs = current_model.generate(
        **inputs,
        do_sample=True,
        temperature=temp,
        max_new_tokens=max_new_tokens
    )

    response = current_tokenizer.decode(
        outputs[0][input_length:],  # slice off the prompt
        skip_special_tokens=True
    )
    return response

# ==============================================================================
# CORE GUESSER PIPELINE CLASS
# ==============================================================================

def infer_answer(question, comp_id) -> tuple:
        try:
            options = [opt.text for opt in question.options]

            # Initialize metrics for this question
            search_doc_time = 0.0
            prefilter_time = 0.0
            rerank_doc_time = 0.0
            reasoning_time = 0.0

            t0_question_start = time.time()

            if comp_id == 3:
                model_prompt = build_math_prompt(question.text, options)
                t_call_model_start = time.time()
                response = call_model(model_prompt, 0.1, False, 1024)
                reasoning_time = time.time() - t_call_model_start
            else:
                model_prompt = build_query_prompt(question.text, options)
                raw_queries = call_model(model_prompt, 1, False, 120)
                #print(f"Raw queries: {raw_queries}")
                queries = [clean_query(q) for q in raw_queries.strip().split("\n")]
                queries = [q for q in queries if 3 < len(q) < 120]
                queries = list(dict.fromkeys(queries))[
                    :3
                ]  # hardcoding of just 3 queries
                print(f"Queries: {queries}")

                # Call wikisearch_multi and get all timings
                found_docs, search_doc_time, prefilter_time, rerank_doc_time = wikisearch_multi(queries, question.text, options)

                #print(f"Retrieved chunks: {found_docs}")
                model_prompt = build_doc_prompt(
                    question.text, options, found_docs
                )
                t_call_model_start = time.time()
                response = call_model(model_prompt, 1, False, 80)
                reasoning_time = time.time() - t_call_model_start

            print(f"Model answered: {response}")

            letter_to_index = {"A": 0, "B": 1, "C": 2, "D": 3}
            match = re.search(r"FINAL_ANSWER:\s*([ABCD])", response)

            answer_index = -1
            if match:
                answer_index = letter_to_index[match.group(1)]
            else:
                answer_index = random.randint(0, 3)

            metrics = {
                "search_doc_time": search_doc_time,
                "prefilter_time": prefilter_time,
                "rerank_doc_time": rerank_doc_time,
                "reasoning_time": reasoning_time,
            }

            return answer_index, metrics

        except Exception as e:
            print(f"Error in infer_answer: {e}")
            metrics = {
                "search_doc_time": 0.0,
                "prefilter_time": 0.0,
                "rerank_doc_time": 0.0,
                "reasoning_time": 0.0,
            }
            return random.randint(0, 3), metrics # Return a random answer and zero metrics on error


# Run game

In [ ]:
#Play game
def game_start(comp_id):
    print("\n=== Starting Game ===")
    game = client.game.start(competition_id=comp_id)
    print(f"Session ID: {game.session_id}")
    print(f"Total number of questions: {game.state.competition.max_levels}")
    print()
    return play_game(game, comp_id)

all_stats = []

# Assuming comp_id 0, 1, 2 use RAG and comp_id 3 uses Math model
for comp_id in range(0, 4):
    model_to_load = rag_model if comp_id < 3 else math_model
    load_model(model_to_load)

    # Number of runs for each competition
    for run in range(1, 4): # Example: 3 runs per competition
        print(f"\n>>> Competition {comp_id} | Run {run}")
        game_metrics = game_start(comp_id)

        # Add common and competition-specific metrics
        stats_entry = {
            "comp_id": comp_id,
            "run": run,
            "model_used": model_to_load,
            "level_reached": game_metrics["level_reached"],
            "avg_response_time": game_metrics["avg_response_time"],
            "accuracy_per_run": game_metrics["accuracy_per_run"]
        }

        # Add RAG-specific metrics if applicable
        if comp_id < 3:
            stats_entry["avg_search_doc_time"] = game_metrics["avg_search_doc_time"]
            stats_entry["avg_prefilter_time"] = game_metrics["avg_prefilter_time"]
            stats_entry["avg_rerank_doc_time"] = game_metrics["avg_rerank_doc_time"]
            stats_entry["avg_reasoning_time"] = game_metrics["avg_reasoning_time"]
        else:
            # For Math model, these times are 0 or not applicable
            stats_entry["avg_search_doc_time"] = 0.0
            stats_entry["avg_prefilter_time"] = 0.0
            stats_entry["avg_rerank_doc_time"] = 0.0
            stats_entry["avg_reasoning_time"] = game_metrics["avg_reasoning_time"] # Reasoning time still applies

        all_stats.append(stats_entry)

# Print summary and save to CSV
import pandas as pd
df = pd.DataFrame(all_stats)
print("\n--- Experiment Results Summary ---")
print(df)

# Calculate total accuracy across all runs and categories if desired
# For simplicity, let's just save the per-run stats for now.
# You can add more complex aggregation here.

df.to_csv("experiment_results.csv", index=False)
print("\nResults saved to experiment_results.csv")

# You might want to display overall average accuracies for each comp_id
overall_accuracy_by_comp = df.groupby('comp_id')['accuracy_per_run'].mean().reset_index()
print("\n--- Overall Average Accuracy by Competition ---")
print(overall_accuracy_by_comp)


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]